In [ ]:
import sqlite3
import pandas as pd

# connect to database file
conn = sqlite3.connect("../data/nfl.db")  # adjust the path if notebook isn't in notebooks/

# load the three tables we need for this step
teams = pd.read_sql("SELECT team_abbr, team_id, team_conf, team_division FROM teams", conn)
games = pd.read_sql("SELECT * FROM games", conn)
pgs   = pd.read_sql("SELECT * FROM player_game_stats", conn)

# quick peek to make sure everything loaded
print(teams.shape)
print(games.shape)
print(pgs.shape)

(36, 4)
(3300, 25)
(54479, 35)


In [2]:
# build a lookup: abbreviation -> stable team_id
abbr_to_id = dict(zip(teams["team_abbr"], teams["team_id"]))

# apply it to player_game_stats (their own team, and their opponent)
pgs["team_id"] = pgs["recent_team"].map(abbr_to_id)
pgs["opp_id"] = pgs["opponent_team"].map(abbr_to_id)

# apply it to games (home team, and away team)
games["home_id"] = games["home_team"].map(abbr_to_id)
games["away_id"] = games["away_team"].map(abbr_to_id)

# check for any abbreviations that failed to map (should print empty arrays)
print("Unmapped in pgs (recent_team):", pgs.loc[pgs["team_id"].isna(), "recent_team"].unique())
print("Unmapped in pgs (opponent_team):", pgs.loc[pgs["opp_id"].isna(), "opponent_team"].unique())
print("Unmapped in games (home_team):", games.loc[games["home_id"].isna(), "home_team"].unique())
print("Unmapped in games (away_team):", games.loc[games["away_id"].isna(), "away_team"].unique())

Unmapped in pgs (recent_team): []
Unmapped in pgs (opponent_team): []
Unmapped in games (home_team): []
Unmapped in games (away_team): []


In [ ]:
# columns to sum up from player-level to team-level
agg_cols = {
    "completions": "sum",
    "attempts": "sum",
    "passing_yards": "sum",
    "passing_tds": "sum",
    "passing_epa": "sum",
    "interceptions": "sum",
    "sacks": "sum",
    "sack_fumbles_lost": "sum",
    "carries": "sum",
    "rushing_yards": "sum",
    "rushing_tds": "sum",
    "rushing_epa": "sum",
    "rushing_fumbles_lost": "sum",
    "receptions": "sum",
    "targets": "sum",
    "receiving_yards": "sum",
    "receiving_tds": "sum",
    "receiving_epa": "sum",
    "receiving_fumbles_lost": "sum",
}

team_off = pgs.groupby(["game_id", "team_id"], as_index=False).agg(agg_cols)

print(team_off.shape)
team_off.head()

(5486, 21)


,game_id,team_id,completions,attempts,passing_yards,passing_tds,passing_epa,interceptions,sacks,sack_fumbles_lost,...,rushing_yards,rushing_tds,rushing_epa,rushing_fumbles_lost,receptions,targets,receiving_yards,receiving_tds,receiving_epa,receiving_fumbles_lost
0,2015_01_BAL_DEN,325,18,32,117.0,0,-17.578054,2.0,2.0,0,...,73.0,0,-4.362854,0.0,18,32,117.0,0,-13.765043,0.0
1,2015_01_BAL_DEN,1400,24,40,175.0,0,-12.047624,1.0,4.0,0,...,69.0,0,-4.829149,0.0,24,40,175.0,0,-6.171885,0.0
2,2015_01_CAR_JAX,750,18,31,175.0,1,-0.279408,1.0,2.0,0,...,105.0,0,-5.129245,0.0,18,31,175.0,1,3.021950,0.0
3,2015_01_CAR_JAX,2250,22,40,183.0,1,-15.522408,2.0,5.0,0,...,96.0,0,-0.896787,0.0,22,39,183.0,1,-13.447137,1.0
4,2015_01_CIN_OAK,920,25,34,269.0,2,13.905390,0.0,0.0,0,...,127.0,2,0.289956,0.0,25,34,269.0,2,13.905390,0.0


In [ ]:
# columns from `games` that don't depend on home/away perspective
shared_cols = ["game_id", "season", "week", "game_type", "gameday",
               "div_game", "roof", "surface", "temp", "wind", "referee", "stadium"]

# home teams perspective
home = games[shared_cols + ["home_id", "away_id", "home_score", "away_score",
                             "home_rest", "away_rest"]].copy()

home = home.rename(columns={
    "home_id": "team_id",
    "away_id": "opp_id",
    "home_score": "points_for",
    "away_score": "points_against",
    "home_rest": "rest_days",
    "away_rest": "opp_rest_days",
})
home["is_home"] = 1

print(home.shape)
home.head()

(3300, 19)


,game_id,season,week,game_type,gameday,div_game,roof,surface,temp,wind,referee,stadium,team_id,opp_id,points_for,points_against,rest_days,opp_rest_days,is_home
0,2015_01_PIT_NE,2015,1,REG,2015-09-10,0,outdoors,fieldturf,65.0,7.0,Carl Cheffers,Gillette Stadium,3200,3900,28.0,21.0,7,7,1
1,2015_01_IND_BUF,2015,1,REG,2015-09-13,0,outdoors,a_turf,56.0,15.0,John Parry,Ralph Wilson Stadium,610,2200,27.0,14.0,7,7,1
2,2015_01_GB_CHI,2015,1,REG,2015-09-13,1,outdoors,grass,72.0,11.0,Craig Wrolstad,Soldier Field,810,1800,23.0,31.0,7,7,1
3,2015_01_KC_HOU,2015,1,REG,2015-09-13,0,closed,grass,NaN,NaN,Peter Morelli,NRG Stadium,2120,2310,20.0,27.0,7,7,1
4,2015_01_CAR_JAX,2015,1,REG,2015-09-13,0,outdoors,grass,77.0,7.0,Ron Torbert,EverBank Field,2250,750,9.0,20.0,7,7,1


In [ ]:
# away teams perspective
away = games[shared_cols + ["home_id", "away_id", "home_score", "away_score",
                             "home_rest", "away_rest"]].copy()

away = away.rename(columns={
    "away_id": "team_id",
    "home_id": "opp_id",
    "away_score": "points_for",
    "home_score": "points_against",
    "away_rest": "rest_days",
    "home_rest": "opp_rest_days",
})
away["is_home"] = 0

# stack home and away rows into one long table
team_games = pd.concat([home, away], ignore_index=True)
team_games = team_games.sort_values(["team_id", "season", "week"]).reset_index(drop=True)

print(team_games.shape)
team_games.head()

(6600, 19)


,game_id,season,week,game_type,gameday,div_game,roof,surface,temp,wind,referee,stadium,team_id,opp_id,points_for,points_against,rest_days,opp_rest_days,is_home
0,2015_01_PHI_ATL,2015,1,REG,2015-09-14,0,dome,fieldturf,NaN,NaN,Terry McAulay,Georgia Dome,200,3700,26.0,24.0,7,7,1
1,2015_02_ATL_NYG,2015,2,REG,2015-09-20,0,outdoors,fieldturf,73.0,9.0,Brad Allen,MetLife Stadium,200,3410,24.0,20.0,6,7,0
2,2015_03_ATL_DAL,2015,3,REG,2015-09-27,0,closed,matrixturf,NaN,NaN,Peter Morelli,AT&T Stadium,200,1200,39.0,28.0,7,7,0
3,2015_04_HOU_ATL,2015,4,REG,2015-10-04,0,dome,fieldturf,NaN,NaN,Carl Cheffers,Georgia Dome,200,2120,48.0,21.0,7,7,1
4,2015_05_WAS_ATL,2015,5,REG,2015-10-11,0,dome,fieldturf,NaN,NaN,Bill Vinovich,Georgia Dome,200,5110,25.0,19.0,7,7,1


In [6]:
team_games = team_games.merge(team_off, on=["game_id", "team_id"], how="left")

print(team_games.shape)
team_games.head()

(6600, 38)


,game_id,season,week,game_type,gameday,div_game,roof,surface,temp,wind,...,rushing_yards,rushing_tds,rushing_epa,rushing_fumbles_lost,receptions,targets,receiving_yards,receiving_tds,receiving_epa,receiving_fumbles_lost
0,2015_01_PHI_ATL,2015,1,REG,2015-09-14,0,dome,fieldturf,NaN,NaN,...,105.0,0.0,-8.790971,0.0,23.0,34.0,298.0,2.0,3.357505,0.0
1,2015_02_ATL_NYG,2015,2,REG,2015-09-20,0,outdoors,fieldturf,73.0,9.0,...,56.0,2.0,-1.516500,0.0,30.0,46.0,363.0,1.0,20.345394,0.0
2,2015_03_ATL_DAL,2015,3,REG,2015-09-27,0,closed,matrixturf,NaN,NaN,...,158.0,3.0,6.369161,0.0,24.0,35.0,285.0,2.0,14.646152,0.0
3,2015_04_HOU_ATL,2015,4,REG,2015-10-04,0,dome,fieldturf,NaN,NaN,...,135.0,4.0,2.149674,0.0,19.0,27.0,256.0,1.0,13.418376,0.0
4,2015_05_WAS_ATL,2015,5,REG,2015-10-11,0,dome,fieldturf,NaN,NaN,...,176.0,1.0,7.312295,0.0,24.0,42.0,254.0,0.0,-0.865304,0.0


In [ ]:
# rename team_off's columns so its clear these are "what the opponent did" 
opp_off = team_off.rename(columns={"team_id": "opp_id"})
opp_off = opp_off.rename(columns={
    col: f"def_{col}" for col in opp_off.columns if col not in ["game_id", "opp_id"]
})

# merge onto team_games matching this team's opponent in the same game
team_games = team_games.merge(opp_off, on=["game_id", "opp_id"], how="left")

print(team_games.shape)
team_games.head()

(6600, 57)


,game_id,season,week,game_type,gameday,div_game,roof,surface,temp,wind,...,def_rushing_yards,def_rushing_tds,def_rushing_epa,def_rushing_fumbles_lost,def_receptions,def_targets,def_receiving_yards,def_receiving_tds,def_receiving_epa,def_receiving_fumbles_lost
0,2015_01_PHI_ATL,2015,1,REG,2015-09-14,0,dome,fieldturf,NaN,NaN,...,63.0,2.0,0.786439,0.0,36.0,52.0,336.0,1.0,4.614013,0.0
1,2015_02_ATL_NYG,2015,2,REG,2015-09-20,0,outdoors,fieldturf,73.0,9.0,...,97.0,0.0,-5.017808,0.0,27.0,40.0,292.0,2.0,12.715258,0.0
2,2015_03_ATL_DAL,2015,3,REG,2015-09-27,0,closed,matrixturf,NaN,NaN,...,127.0,4.0,3.571723,0.0,22.0,26.0,232.0,0.0,5.148783,0.0
3,2015_04_HOU_ATL,2015,4,REG,2015-10-04,0,dome,fieldturf,NaN,NaN,...,54.0,1.0,-9.063123,1.0,29.0,57.0,382.0,2.0,-4.903847,2.0
4,2015_05_WAS_ATL,2015,5,REG,2015-10-11,0,dome,fieldturf,NaN,NaN,...,51.0,1.0,-4.429095,0.0,21.0,32.0,219.0,1.0,-6.220102,0.0


In [8]:
team_games["total_epa"] = (
    team_games["passing_epa"].fillna(0)
    + team_games["rushing_epa"].fillna(0)
    + team_games["receiving_epa"].fillna(0)
)

team_games["def_total_epa"] = (
    team_games["def_passing_epa"].fillna(0)
    + team_games["def_rushing_epa"].fillna(0)
    + team_games["def_receiving_epa"].fillna(0)
)

team_games["total_yards"] = team_games["passing_yards"] + team_games["rushing_yards"]
team_games["def_total_yards"] = team_games["def_passing_yards"] + team_games["def_rushing_yards"]

print(team_games[["total_epa", "def_total_epa", "total_yards", "def_total_yards"]].describe())

         total_epa  def_total_epa  total_yards  def_total_yards
count  6600.000000    6600.000000  5486.000000      5486.000000
mean      4.931107       4.931107   360.919067       360.919067
std      19.032206      19.032206    80.344479        80.344479
min     -74.869878     -74.869878    99.000000        99.000000
25%      -4.482347      -4.482347   305.000000       305.000000
50%       0.994298       0.994298   362.000000       362.000000
75%      17.229484      17.229484   414.000000       414.000000
max      81.549738      81.549738   726.000000       726.000000


In [ ]:
roll_stats = [
    "points_for", "points_against",
    "total_epa", "def_total_epa",
    "total_yards", "def_total_yards",
]

# Group by team so rolling calculations happen within each team's own game history
grouped = team_games.groupby("team_id")

for col in roll_stats:
    shifted = grouped[col].shift(1)  # exclude current game from its own rolling stats

    team_games[f"{col}_roll3"] = shifted.groupby(team_games["team_id"]).transform(
        lambda x: x.rolling(3, min_periods=1).mean()
    )
    team_games[f"{col}_roll8"] = shifted.groupby(team_games["team_id"]).transform(
        lambda x: x.rolling(8, min_periods=1).mean()
    )
    team_games[f"{col}_season"] = shifted.groupby(team_games["season"].astype(str) + "_" + team_games["team_id"].astype(str)).transform(
        lambda x: x.expanding(min_periods=1).mean()
    )

print(team_games.shape)
team_games[["team_id", "season", "week", "points_for", "points_for_roll3", "points_for_roll8", "points_for_season"]].head(15)

(6600, 79)


,team_id,season,week,points_for,points_for_roll3,points_for_roll8,points_for_season
0,200,2015,1,26.0,NaN,NaN,NaN
1,200,2015,2,24.0,26.000000,26.000000,26.000000
2,200,2015,3,39.0,25.000000,25.000000,25.000000
3,200,2015,4,48.0,29.666667,29.666667,29.666667
4,200,2015,5,25.0,37.000000,34.250000,34.250000
5,200,2015,6,21.0,37.333333,32.400000,32.400000
6,200,2015,7,10.0,31.333333,30.500000,30.500000
7,200,2015,8,20.0,18.666667,27.571429,27.571429
8,200,2015,9,16.0,17.000000,26.625000,26.625000
9,200,2015,11,21.0,15.333333,25.375000,25.444444


In [ ]:
# get one current abbreviation per team_id 
team_display = teams.groupby("team_id", as_index=False)["team_abbr"].last()
team_display = team_display.rename(columns={"team_abbr": "team_abbr_current"})

team_games = team_games.merge(team_display, on="team_id", how="left")

print(team_games.shape)
team_games[["team_id", "team_abbr_current", "season", "week", "points_for", 
            "points_for_roll3", "points_for_roll8", "points_for_season"]].head(15)

(6600, 80)


,team_id,team_abbr_current,season,week,points_for,points_for_roll3,points_for_roll8,points_for_season
0,200,ATL,2015,1,26.0,NaN,NaN,NaN
1,200,ATL,2015,2,24.0,26.000000,26.000000,26.000000
2,200,ATL,2015,3,39.0,25.000000,25.000000,25.000000
3,200,ATL,2015,4,48.0,29.666667,29.666667,29.666667
4,200,ATL,2015,5,25.0,37.000000,34.250000,34.250000
5,200,ATL,2015,6,21.0,37.333333,32.400000,32.400000
6,200,ATL,2015,7,10.0,31.333333,30.500000,30.500000
7,200,ATL,2015,8,20.0,18.666667,27.571429,27.571429
8,200,ATL,2015,9,16.0,17.000000,26.625000,26.625000
9,200,ATL,2015,11,21.0,15.333333,25.375000,25.444444


In [11]:
raiders_id = team_games.loc[team_games["team_abbr_current"].isin(["OAK", "LV"]), "team_id"].unique()
print(raiders_id)
teams[teams["team_id"].isin(raiders_id)]

[2520]


,team_abbr,team_id,team_conf,team_division
19,LV,2520,AFC,AFC West
26,OAK,2520,AFC,AFC West


In [12]:
# grab just the columns we need from the "opponent's own perspective" rows
opp_def_lookup = team_games[["game_id", "team_id", "def_total_epa_roll3", 
                               "def_total_epa_roll8", "def_total_epa_season"]].copy()

opp_def_lookup = opp_def_lookup.rename(columns={
    "team_id": "opp_id",
    "def_total_epa_roll3": "opp_def_epa_roll3",
    "def_total_epa_roll8": "opp_def_epa_roll8",
    "def_total_epa_season": "opp_def_epa_season",
})

team_games = team_games.merge(opp_def_lookup, on=["game_id", "opp_id"], how="left")

print(team_games.shape)
team_games[["team_abbr_current", "season", "week", "total_epa_roll8", 
            "opp_def_epa_roll8"]].head(10)

(6600, 83)


,team_abbr_current,season,week,total_epa_roll8,opp_def_epa_roll8
0,ATL,2015,1,NaN,NaN
1,ATL,2015,2,-4.119131,6.383988
2,ATL,2015,3,15.965778,-9.994662
3,ATL,2015,4,21.939962,-2.679747
4,ATL,2015,5,22.758833,6.897304
5,ATL,2015,6,17.914883,18.176621
6,ATL,2015,7,17.641048,3.891755
7,ATL,2015,8,13.314879,12.855108
8,ATL,2015,9,14.515732,16.236867
9,ATL,2015,11,14.981196,7.479402


In [13]:
# adjusted offensive performance = how much better/worse this team's typical offense is
# compared to what this specific opponent's defense normally allows
team_games["adj_epa_roll8"] = team_games["total_epa_roll8"] - team_games["opp_def_epa_roll8"]

team_games[["team_abbr_current", "season", "week", "total_epa_roll8", 
            "opp_def_epa_roll8", "adj_epa_roll8"]].head(10)

,team_abbr_current,season,week,total_epa_roll8,opp_def_epa_roll8,adj_epa_roll8
0,ATL,2015,1,NaN,NaN,NaN
1,ATL,2015,2,-4.119131,6.383988,-10.503119
2,ATL,2015,3,15.965778,-9.994662,25.960439
3,ATL,2015,4,21.939962,-2.679747,24.619709
4,ATL,2015,5,22.758833,6.897304,15.861529
5,ATL,2015,6,17.914883,18.176621,-0.261738
6,ATL,2015,7,17.641048,3.891755,13.749292
7,ATL,2015,8,13.314879,12.855108,0.459770
8,ATL,2015,9,14.515732,16.236867,-1.721135
9,ATL,2015,11,14.981196,7.479402,7.501794


In [14]:
team_games.query("team_id == 200 and season == 2015 and week in [3,4]")[
    ["team_abbr_current", "season", "week", "opp_id", "points_for", "points_against", "adj_epa_roll8"]
]

,team_abbr_current,season,week,opp_id,points_for,points_against,adj_epa_roll8
2,ATL,2015,3,1200,39.0,28.0,25.960439
3,ATL,2015,4,2120,48.0,21.0,24.619709


In [15]:
# --- offensive adjustment, for the other two time windows ---
team_games["adj_epa_roll3"] = team_games["total_epa_roll3"] - team_games["opp_def_epa_roll3"]
team_games["adj_epa_season"] = team_games["total_epa_season"] - team_games["opp_def_epa_season"]

# --- now the defensive mirror: how much worse/better is this team's defense, 
#     compared to what this opponent's offense normally produces? ---

# first, grab the opponent's own OFFENSIVE rolling numbers (not defensive) for this game
opp_off_lookup = team_games[["game_id", "team_id", "total_epa_roll3", 
                               "total_epa_roll8", "total_epa_season"]].copy()

opp_off_lookup = opp_off_lookup.rename(columns={
    "team_id": "opp_id",
    "total_epa_roll3": "opp_off_epa_roll3",
    "total_epa_roll8": "opp_off_epa_roll8",
    "total_epa_season": "opp_off_epa_season",
})

team_games = team_games.merge(opp_off_lookup, on=["game_id", "opp_id"], how="left")

# defensive adjustment = opponent's normal offensive output minus what this team's defense normally allows
# positive = this defense is BETTER than average against this caliber of offense (holding them below their norm)
team_games["adj_def_epa_roll3"] = team_games["opp_off_epa_roll3"] - team_games["def_total_epa_roll3"]
team_games["adj_def_epa_roll8"] = team_games["opp_off_epa_roll8"] - team_games["def_total_epa_roll8"]
team_games["adj_def_epa_season"] = team_games["opp_off_epa_season"] - team_games["def_total_epa_season"]

print(team_games.shape)
team_games[["team_abbr_current", "season", "week", "def_total_epa_roll8", 
            "opp_off_epa_roll8", "adj_def_epa_roll8"]].head(10)

(6600, 92)


,team_abbr_current,season,week,def_total_epa_roll8,opp_off_epa_roll8,adj_def_epa_roll8
0,ATL,2015,1,NaN,NaN,NaN
1,ATL,2015,2,10.014465,7.041491,-2.972975
2,ATL,2015,3,13.746013,5.161988,-8.584026
3,ATL,2015,4,12.602900,-9.709742,-22.312643
4,ATL,2015,5,7.128241,6.691957,-0.436284
5,ATL,2015,6,2.152496,7.773304,5.620808
6,ATL,2015,7,5.941846,4.423636,-1.518209
7,ATL,2015,8,1.198299,2.169830,0.971532
8,ATL,2015,9,1.584183,-4.081369,-5.665551
9,ATL,2015,11,0.868374,0.051856,-0.816519


In [16]:
# rest advantage: positive = this team had more rest than their opponent (a real, sometimes meaningful edge)
team_games["rest_advantage"] = team_games["rest_days"] - team_games["opp_rest_days"]

# short week flag: teams on a short week (e.g. Thursday Night Football) are often at a disadvantage
team_games["short_week"] = (team_games["rest_days"] < 6).astype(int)

# indoor game flag: roof values include "dome", "closed", "outdoors", "open" -- 
# treat dome/closed as indoor (no real weather impact), outdoors/open as outdoor
team_games["is_indoor"] = team_games["roof"].isin(["dome", "closed"]).astype(int)

# missing temp/wind almost always means it was an indoor game -- fill with a neutral value
# rather than leaving NaN (NaN would cause problems for most ML models later)
team_games["temp"] = team_games["temp"].fillna(70)   # 70°F = neutral indoor temp
team_games["wind"] = team_games["wind"].fillna(0)    # no wind indoors

# divisional flag is already in the data as div_game (0/1) -- nothing to change there,
# just noting it's already usable as-is

print(team_games[["rest_advantage", "short_week", "is_indoor", "temp", "wind", "div_game"]].describe())

       rest_advantage   short_week    is_indoor         temp         wind  \
count     6600.000000  6600.000000  6600.000000  6600.000000  6600.000000   
mean         0.000000     0.061818     0.274848    62.546061     4.996061   
std          2.505578     0.240843     0.446472    14.954231     5.678857   
min         -8.000000     0.000000     0.000000    -6.000000     0.000000   
25%          0.000000     0.000000     0.000000    54.000000     0.000000   
50%          0.000000     0.000000     0.000000    70.000000     4.000000   
75%          0.000000     0.000000     1.000000    70.000000     8.250000   
max          8.000000     1.000000     1.000000    97.000000    71.000000   

          div_game  
count  6600.000000  
mean      0.354545  
std       0.478412  
min       0.000000  
25%       0.000000  
50%       0.000000  
75%       1.000000  
max       1.000000  


In [17]:
depth_charts_hist = pd.read_sql("SELECT * FROM depth_charts_historical", conn)
injuries = pd.read_sql("SELECT * FROM injuries", conn)

print(depth_charts_hist["depth_team"].value_counts().head(10))
print(depth_charts_hist["position"].value_counts().head(15))
print(injuries["report_status"].value_counts())

1    176728
2    138523
3     53981
Name: depth_team, dtype: int64
WR     54919
CB     36497
RB     31181
T      24802
G      23869
OLB    23575
TE     23217
DE     22939
DT     18838
ILB    18525
QB     17445
FS     16810
SS     15422
C      14608
P      14018
Name: position, dtype: int64
Questionable    16040
Out             11250
Probable         2702
Doubtful         1889
Note                6
Name: report_status, dtype: int64


In [18]:
# check if depth_team == "1" for QB/RB/WR ever has more than one player per team per week
check = depth_charts_hist[
    (depth_charts_hist["depth_team"] == "1") & 
    (depth_charts_hist["position"].isin(["QB", "RB", "WR"]))
]

dupes = check.groupby(["season", "week", "club_code", "position"]).size().reset_index(name="count")
print(dupes["count"].value_counts())

1     9162
2     3099
4     2621
3     2198
5     1102
8       44
6       29
10      19
7        5
9        1
Name: count, dtype: int64


In [19]:
dupes_by_pos = check.groupby(["season", "week", "club_code", "position"]).size().reset_index(name="count")
print(dupes_by_pos.groupby("position")["count"].value_counts())

position  count
QB        1        5895
          2         192
          3           1
          4           1
RB        1        3234
          2        2086
          3         684
          4          65
          6           6
          5           3
          7           1
WR        4        2555
          3        1513
          5        1099
          2         821
          8          44
          1          33
          6          23
          10         19
          7           4
          9           1
Name: count, dtype: int64


In [20]:
print(depth_charts_hist["formation"].value_counts())

Offense          157135
Defense          156160
Special Teams     55937
Name: formation, dtype: int64


In [21]:
print(depth_charts_hist[depth_charts_hist["position"] == "WR"]["depth_position"].value_counts())
print(depth_charts_hist[depth_charts_hist["position"] == "RB"]["depth_position"].value_counts())
print(depth_charts_hist[depth_charts_hist["position"] == "QB"]["depth_position"].value_counts())

WR               33691
PR                9527
KR                5275
Wide Receiver     3912
KOR               1345
\n                 671
H                   98
RB                  73
LWR                 64
RWR                 61
CB                  43
FB                  25
WR2                 24
RCB                 24
WRE                 20
WR1                 16
QB                  12
TE                  10
WR\8                 8
KO                   5
PF                   5
19                   5
LS                   3
LCB                  1
LT                   1
Name: depth_position, dtype: int64
RB              17497
KR               4090
\n               2491
Running Back     2433
PR               1570
HB               1338
KOR              1102
FB                370
WR                189
J                  19
CB                 19
LS                 18
FS                  7
QB                  7
PF                  6
TE                  4
19                  3
SS              

In [22]:
# get all "starters" (depth_team == 1) at the 3 positions we care about
starters = depth_charts_hist[
    (depth_charts_hist["depth_team"] == "1") &
    (depth_charts_hist["position"].isin(["QB", "RB", "WR"]))
][["season", "week", "club_code", "position", "gsis_id"]].copy()

# get everyone marked "Out" that week
outs = injuries[injuries["report_status"] == "Out"][
    ["season", "week", "team", "gsis_id"]
].copy()

# merge: for each starter, check if they show up in the "Out" list that same team/week
starters_out = starters.merge(
    outs,
    left_on=["season", "week", "club_code", "gsis_id"],
    right_on=["season", "week", "team", "gsis_id"],
    how="left",
    indicator=True
)

starters_out["is_out"] = (starters_out["_merge"] == "both").astype(int)

# collapse to team/week/position level: 1 if ANY starter at that position is out
position_flags = starters_out.groupby(["season", "week", "club_code", "position"])["is_out"].max().reset_index()

# pivot so each position becomes its own column
position_flags_wide = position_flags.pivot(
    index=["season", "week", "club_code"], columns="position", values="is_out"
).reset_index()

position_flags_wide = position_flags_wide.rename(columns={
    "QB": "qb_out", "RB": "rb_starter_out", "WR": "wr_starter_out"
})

print(position_flags_wide.shape)
position_flags_wide.head(10)

(6116, 6)


position,season,week,club_code,qb_out,rb_starter_out,wr_starter_out
0,2015,1.0,ARI,0.0,0.0,0.0
1,2015,1.0,ATL,0.0,0.0,0.0
2,2015,1.0,BAL,0.0,0.0,0.0
3,2015,1.0,BUF,0.0,0.0,0.0
4,2015,1.0,CAR,0.0,0.0,0.0
5,2015,1.0,CHI,0.0,0.0,0.0
6,2015,1.0,CIN,0.0,0.0,0.0
7,2015,1.0,CLE,0.0,0.0,0.0
8,2015,1.0,DAL,0.0,0.0,0.0
9,2015,1.0,DEN,0.0,0.0,0.0


In [23]:
print(position_flags_wide.shape)
position_flags_wide[["qb_out", "rb_starter_out", "wr_starter_out"]].sum()

(6116, 6)


position
qb_out            180.0
rb_starter_out    320.0
wr_starter_out    666.0
dtype: float64

In [24]:
# map club_code (season-accurate abbreviation) to the stable team_id -- same lookup we've used all along
position_flags_wide["team_id"] = position_flags_wide["club_code"].map(abbr_to_id)

# merge onto team_games using team_id (safe across relocations) instead of abbreviation
team_games = team_games.merge(
    position_flags_wide[["season", "week", "team_id", "qb_out", "rb_starter_out", "wr_starter_out"]],
    on=["season", "week", "team_id"],
    how="left"
)

# games with no matching injury data (e.g. 2026 future games, or weeks with no report) -> assume 0 (healthy/unknown)
team_games["qb_out"] = team_games["qb_out"].fillna(0).astype(int)
team_games["rb_starter_out"] = team_games["rb_starter_out"].fillna(0).astype(int)
team_games["wr_starter_out"] = team_games["wr_starter_out"].fillna(0).astype(int)

print(team_games.shape)
team_games[["team_abbr_current", "season", "week", "qb_out", "rb_starter_out", "wr_starter_out"]].query("qb_out == 1").head(10)

(6600, 98)


,team_abbr_current,season,week,qb_out,rb_starter_out,wr_starter_out
319,BAL,2021,18,1,0,0
354,BAL,2023,18,1,0,1
455,BUF,2017,15,1,0,0
465,BUF,2018,7,1,0,0
646,CAR,2016,5,1,0,0
689,CAR,2018,16,1,0,0
690,CAR,2018,17,1,0,0
693,CAR,2019,3,1,0,0
694,CAR,2019,4,1,0,0
750,CAR,2022,11,1,0,0


In [25]:
print(team_games.shape)
print(team_games.columns.tolist())

(6600, 98)
['game_id', 'season', 'week', 'game_type', 'gameday', 'div_game', 'roof', 'surface', 'temp', 'wind', 'referee', 'stadium', 'team_id', 'opp_id', 'points_for', 'points_against', 'rest_days', 'opp_rest_days', 'is_home', 'completions', 'attempts', 'passing_yards', 'passing_tds', 'passing_epa', 'interceptions', 'sacks', 'sack_fumbles_lost', 'carries', 'rushing_yards', 'rushing_tds', 'rushing_epa', 'rushing_fumbles_lost', 'receptions', 'targets', 'receiving_yards', 'receiving_tds', 'receiving_epa', 'receiving_fumbles_lost', 'def_completions', 'def_attempts', 'def_passing_yards', 'def_passing_tds', 'def_passing_epa', 'def_interceptions', 'def_sacks', 'def_sack_fumbles_lost', 'def_carries', 'def_rushing_yards', 'def_rushing_tds', 'def_rushing_epa', 'def_rushing_fumbles_lost', 'def_receptions', 'def_targets', 'def_receiving_yards', 'def_receiving_tds', 'def_receiving_epa', 'def_receiving_fumbles_lost', 'total_epa', 'def_total_epa', 'total_yards', 'def_total_yards', 'points_for_roll3'

In [26]:
team_games.to_sql("team_games", conn, if_exists="replace", index=False)
print("Saved. Row count in DB:", pd.read_sql("SELECT COUNT(*) FROM team_games", conn).iloc[0,0])

Saved. Row count in DB: 6600
